# Asset Embeddings: Robustness and Sanity Checks

This notebook audits the completed 80-quarter replication without altering its baseline artifacts. It tests target construction, alternative firm splits, transductive versus inductive representation learning, RS-L regularization, Word2Vec calibration, and economic nearest-neighbor behavior. All new outputs are stored under `full_replication_80q/robustness`.


## 1. Environment and immutable artifact paths


In [ ]:
from pathlib import Path
import gc, sys, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != 'Asset_Embeddings':
    candidate = PROJECT_ROOT / 'Projects' / 'Asset_Embeddings'
    if candidate.exists(): PROJECT_ROOT = candidate.resolve()
if PROJECT_ROOT.name != 'Asset_Embeddings': raise RuntimeError('Open from the Asset_Embeddings project directory.')
FULL_ROOT = PROJECT_ROOT / 'full_replication_80q'
RS_ROOT = FULL_ROOT / 'additional_recommender_systems'
W2V_ROOT = FULL_ROOT / 'word2vec'
ROBUST_ROOT = FULL_ROOT / 'robustness'
for folder in ['audits','targets','splits','inductive','rs_l','word2vec','business_checks']:
    (ROBUST_ROOT / folder).mkdir(parents=True, exist_ok=True)
QUARTERS = [(y,q) for y in range(2005,2025) for q in range(1,5)]
DIAGNOSTIC_QUARTERS = [(2005,1),(2014,4),(2024,4)]
RIDGE_ALPHAS = np.logspace(-6,6,49)
BASE_SEED = 202451
def qlabel(y,q): return f'{y}q{q}'
def save(frame,path): path.parent.mkdir(parents=True,exist_ok=True); frame.to_parquet(path,index=False)
print('Baseline artifacts:', FULL_ROOT)
print('Robustness outputs:', ROBUST_ROOT)


## 2. Completion and integrity audit

This verifies that all expected quarterly targets and embeddings exist, contain finite coordinates, and cover the saved evaluation samples.


In [ ]:
def embedding_path(model,y,q,dimension):
    label=qlabel(y,q)
    if model in {'rs_binary','rs_l_min'}:
        return FULL_ROOT/'embeddings'/label/model/'asset_embeddings_normalized_labeled.parquet'
    if model in {'rs_ranks','rs_l_0'}:
        return RS_ROOT/'embeddings'/label/model/'asset_embeddings_normalized_labeled.parquet'
    if model=='rs_l':
        return RS_ROOT/'embeddings'/label/model/f'dimension={dimension}'/'asset_embeddings_labeled.parquet'
    if model=='word2vec':
        return W2V_ROOT/'embeddings'/label/f'dimension={dimension}'/'asset_embeddings_labeled.parquet'
    raise ValueError(model)

audit=[]
for y,q in QUARTERS:
    evaluation=pd.read_parquet(FULL_ROOT/'benchmarks'/qlabel(y,q)/'evaluation_sample_and_split.parquet')
    for model in ['rs_binary','rs_ranks','rs_l_0','rs_l_min','rs_l','word2vec']:
        for dimension in [4,10]:
            p=embedding_path(model,y,q,dimension); frame=pd.read_parquet(p)
            columns=sorted(c for c in frame if c.startswith('embedding_'))[:dimension]
            audit.append({'year':y,'quarter':q,'model':model,'dimension':dimension,'file_exists':p.exists(),
                          'finite':bool(np.isfinite(frame[columns].to_numpy()).all()),
                          'evaluation_coverage':evaluation.permco.isin(frame.permco).mean(),
                          'firms':len(frame)})
integrity=pd.DataFrame(audit)
save(integrity,ROBUST_ROOT/'audits'/'embedding_integrity.parquet')
display(integrity.groupby(['model','dimension'],as_index=False).agg(quarters=('quarter','size'),minimum_coverage=('evaluation_coverage','min'),all_finite=('finite','all')))


## 3. Rebuild and audit the relative-valuation target

The baseline target uses the full quarterly cross section to residualize log market equity on log book equity. We verify it exactly and create a stricter alternative whose residualization coefficients are fitted on valuation-training firms only. Book equity is never included among downstream predictors.


In [ ]:
target_rows=[]
for y,q in QUARTERS:
    p=FULL_ROOT/'benchmarks'/qlabel(y,q)/'evaluation_sample_and_split.parquet'
    d=pd.read_parquet(p); log_be=np.log(d.jkp_book_equity.to_numpy(float)).reshape(-1,1); log_me=np.log(d.jkp_firm_market_equity.to_numpy(float))
    full=LinearRegression().fit(log_be,log_me); rebuilt=log_me-full.predict(log_be)
    train=d['sample'].eq('train').to_numpy(); strict=LinearRegression().fit(log_be[train],log_me[train]); strict_target=log_me-strict.predict(log_be)
    out=d[['permco','sample']].copy(); out['baseline_relative_valuation']=d.relative_valuation; out['strict_relative_valuation']=strict_target
    save(out,ROBUST_ROOT/'targets'/f'{qlabel(y,q)}.parquet')
    target_rows.append({'year':y,'quarter':q,'firms':len(d),'maximum_rebuild_error':np.max(np.abs(rebuilt-d.relative_valuation)),
                        'full_slope':full.coef_[0],'training_only_slope':strict.coef_[0],
                        'target_correlation':np.corrcoef(rebuilt,strict_target)[0,1]})
target_audit=pd.DataFrame(target_rows); save(target_audit,ROBUST_ROOT/'targets'/'target_audit.parquet'); display(target_audit.describe())


## 4. Training-only target benchmark

Every model is re-evaluated with the stricter target while preserving the original firm split. Scaling and ridge selection remain training-only. This isolates target residualization from embedding construction.


In [ ]:
def _closed_form_ridge(X,y,alpha):
    design=np.column_stack([np.ones(len(X)),X]); penalty=np.diag(np.r_[0.0,np.repeat(alpha,X.shape[1])])
    return np.linalg.solve(design.T@design+penalty,design.T@y)

def fast_ridge_result(panel,columns,target,seed=BASE_SEED):
    train=panel['sample'].eq('train').to_numpy(); test=~train; X=panel[columns].to_numpy(float); y=panel[target].to_numpy(float)
    scaler=StandardScaler().fit(X[train]); Xtr=scaler.transform(X[train]); Xte=scaler.transform(X[test]); ytr=y[train]
    folds=list(KFold(10,shuffle=True,random_state=seed).split(Xtr)); losses=[]
    for alpha in RIDGE_ALPHAS:
        squared_error=0.0; observations=0
        for fit_index,validation_index in folds:
            coefficient=_closed_form_ridge(Xtr[fit_index],ytr[fit_index],alpha)
            prediction=np.column_stack([np.ones(len(validation_index)),Xtr[validation_index]])@coefficient
            squared_error+=np.square(ytr[validation_index]-prediction).sum(); observations+=len(validation_index)
        losses.append(squared_error/observations)
    selected=float(RIDGE_ALPHAS[int(np.argmin(losses))]); coefficient=_closed_form_ridge(Xtr,ytr,selected)
    prediction=np.column_stack([np.ones(test.sum()),Xte])@coefficient
    return float(1-np.var(y[test]-prediction)/np.var(y[test])),selected

strict_rows=[]; quarter_directory=ROBUST_ROOT/'targets'/'quarter_results'; quarter_directory.mkdir(parents=True,exist_ok=True)
for y,q in QUARTERS:
    label=qlabel(y,q); checkpoint=quarter_directory/f'{label}.parquet'
    if checkpoint.exists(): strict_rows.append(pd.read_parquet(checkpoint)); print(label,'reused'); continue
    started=time.perf_counter(); panel=pd.read_parquet(FULL_ROOT/'benchmarks'/label/'evaluation_sample_and_split.parquet')
    panel=panel.merge(pd.read_parquet(ROBUST_ROOT/'targets'/f'{label}.parquet')[['permco','strict_relative_valuation']],on='permco',validate='one_to_one')
    specifications={'characteristics_4':['beta_60m_paper','at_gr1_paper','gp_at_paper','div_at_paper']}
    for model in ['rs_binary','rs_ranks','rs_l_0','rs_l_min','rs_l','word2vec']:
        for dimension in [4,10]:
            e=pd.read_parquet(embedding_path(model,y,q,dimension)); cols=sorted(c for c in e if c.startswith('embedding_'))[:dimension]
            renamed={c:f'x_{i+1:03d}' for i,c in enumerate(cols)}; local=panel.merge(e[['permco']+cols].rename(columns=renamed),on='permco',validate='one_to_one')
            r2,alpha=fast_ridge_result(local,list(renamed.values()),'strict_relative_valuation')
            strict_rows.append({'year':y,'quarter':q,'model':f'{model}_{dimension}','test_variance_r2':r2,'selected_alpha':alpha})
    r2,alpha=fast_ridge_result(panel,specifications['characteristics_4'],'strict_relative_valuation')
    strict_rows.append({'year':y,'quarter':q,'model':'characteristics_4','test_variance_r2':r2,'selected_alpha':alpha})
    quarter_frame=pd.DataFrame([row for row in strict_rows if isinstance(row,dict) and row['year']==y and row['quarter']==q]); save(quarter_frame,checkpoint)
    print(label,f'completed in {time.perf_counter()-started:.1f}s'); del panel,quarter_frame; gc.collect()
strict_results=pd.concat([pd.DataFrame([x]) if isinstance(x,dict) else x for x in strict_rows],ignore_index=True); strict_summary=strict_results.groupby('model').test_variance_r2.agg(['mean','std','min','max']).reset_index().sort_values('mean',ascending=False)
save(strict_results,ROBUST_ROOT/'targets'/'training_only_target_results.parquet'); save(strict_summary,ROBUST_ROOT/'targets'/'training_only_target_summary.parquet'); display(strict_summary)


## 5. Alternative firm-split sensitivity

This repeats the valuation exercise across five deterministic 80/20 splits. Embeddings remain fixed, so this tests valuation-split sensitivity rather than representation-learning stability.


In [ ]:
split_rows=[]; split_seeds=[202451,202452,202453,202454,202455]; split_directory=ROBUST_ROOT/'splits'/'quarter_seed_results'; split_directory.mkdir(parents=True,exist_ok=True)
for y,q in QUARTERS:
    label=qlabel(y,q); base=pd.read_parquet(FULL_ROOT/'benchmarks'/label/'evaluation_sample_and_split.parquet').sort_values('permco').reset_index(drop=True)
    model_frames={}
    for model,dimension in [('rs_l_min',10),('rs_binary',10),('word2vec',10)]:
        e=pd.read_parquet(embedding_path(model,y,q,dimension)); cols=sorted(c for c in e if c.startswith('embedding_'))[:dimension]
        model_frames[model]=(e[['permco']+cols],cols)
    for seed in split_seeds:
        checkpoint=split_directory/f'{label}_seed={seed}.parquet'
        if checkpoint.exists(): split_rows.append(pd.read_parquet(checkpoint)); continue
        local_rows=[]; started=time.perf_counter()
        rng=np.random.default_rng(seed+y*10+q); test_ids=set(base.iloc[rng.permutation(len(base))[:int(np.ceil(.2*len(base)))]].permco)
        base['sample']=np.where(base.permco.isin(test_ids),'test','train')
        logbe=np.log(base.jkp_book_equity.to_numpy(float)).reshape(-1,1); logme=np.log(base.jkp_firm_market_equity.to_numpy(float)); tr=base['sample'].eq('train').to_numpy()
        val=LinearRegression().fit(logbe[tr],logme[tr]); base['strict_target']=logme-val.predict(logbe)
        r2,_=fast_ridge_result(base,['beta_60m_paper','at_gr1_paper','gp_at_paper','div_at_paper'],'strict_target',seed); local_rows.append({'year':y,'quarter':q,'seed':seed,'model':'characteristics_4','r2':r2})
        for model,(e,cols) in model_frames.items():
            local=base.merge(e,on='permco',validate='one_to_one'); r2,_=fast_ridge_result(local,cols,'strict_target',seed); local_rows.append({'year':y,'quarter':q,'seed':seed,'model':f'{model}_10','r2':r2})
        local_frame=pd.DataFrame(local_rows); save(local_frame,checkpoint); split_rows.append(local_frame); print(label,seed,f'completed in {time.perf_counter()-started:.1f}s')
    del base,model_frames; gc.collect()
split_results=pd.concat(split_rows,ignore_index=True); split_summary=split_results.groupby(['model','seed'],as_index=False).r2.mean()
save(split_results,ROBUST_ROOT/'splits'/'five_seed_results.parquet'); save(split_summary,ROBUST_ROOT/'splits'/'five_seed_summary.parquet'); display(split_summary.pivot(index='model',columns='seed',values='r2'))


## 6. Strict inductive PCA fold-in pilot

The baseline is intentionally transductive: unsupervised embeddings see holdings of test firms. This pilot excludes valuation-test firms when fitting PCA, then estimates their coordinates from investor holdings with fixed investor scores. It starts with RS-Binary on the three diagnostic quarters; other PCA variants are added only after this design is validated.


In [ ]:
def inductive_binary_foldin(y,q,dimension):
    label=qlabel(y,q); matrix=sparse.load_npz(FULL_ROOT/'matrices'/label/'binary.npz').toarray().astype('float32')
    firms=pd.read_parquet(FULL_ROOT/'matrices'/label/'firm_mapping.parquet').sort_values('firm_index').reset_index(drop=True)
    evaluation=pd.read_parquet(FULL_ROOT/'benchmarks'/label/'evaluation_sample_and_split.parquet')
    test_permcos=set(evaluation.loc[evaluation['sample'].eq('test'),'permco']); test_mask=firms.permco.isin(test_permcos).to_numpy(); train_mask=~test_mask
    pca=PCA(n_components=dimension,svd_solver='randomized',random_state=BASE_SEED).fit(matrix[:,train_mask])
    investor_scores=pca.transform(matrix[:,train_mask]); train_coordinates=pca.components_.T*np.sqrt(train_mask.sum())
    test_values=matrix[:,test_mask]; test_centered=test_values-test_values.mean(axis=0,keepdims=True)
    fold=Ridge(alpha=1e-6,fit_intercept=False).fit(investor_scores,test_centered); test_coordinates=fold.coef_*np.sqrt(train_mask.sum())
    coordinates=np.empty((len(firms),dimension)); coordinates[train_mask]=train_coordinates; coordinates[test_mask]=test_coordinates
    out=firms[['permco','firm_name','firm_index']].copy()
    for k in range(dimension): out[f'embedding_{k+1:03d}']=coordinates[:,k]
    return out

foldin_rows=[]; foldin_directory=ROBUST_ROOT/'inductive'/'quarter_results'; foldin_directory.mkdir(parents=True,exist_ok=True)
for y,q in DIAGNOSTIC_QUARTERS:
    label=qlabel(y,q); panel=pd.read_parquet(FULL_ROOT/'benchmarks'/label/'evaluation_sample_and_split.parquet')
    strict=pd.read_parquet(ROBUST_ROOT/'targets'/f'{label}.parquet')[['permco','strict_relative_valuation']]; panel=panel.merge(strict,on='permco',validate='one_to_one')
    for dimension in [4,10]:
        checkpoint=foldin_directory/f'{label}_dimension={dimension}.parquet'
        if checkpoint.exists(): foldin_rows.append(pd.read_parquet(checkpoint)); continue
        e=inductive_binary_foldin(y,q,dimension); cols=sorted(c for c in e if c.startswith('embedding_')); local=panel.merge(e[['permco']+cols],on='permco',validate='one_to_one')
        r2,alpha=fast_ridge_result(local,cols,'strict_relative_valuation'); result=pd.DataFrame([{'year':y,'quarter':q,'dimension':dimension,'test_variance_r2':r2,'selected_alpha':alpha}]); save(result,checkpoint); foldin_rows.append(result); del e,local; gc.collect()
foldin_results=pd.concat(foldin_rows,ignore_index=True); save(foldin_results,ROBUST_ROOT/'inductive'/'rs_binary_three_quarter_foldin.parquet'); display(foldin_results)


## 7. RS-L regularization-curve audit

This consolidates every saved ten-fold curve and measures the margin between the selected penalty and its nearest competitor. No model is refitted.


In [ ]:
curve_rows=[]
for y,q in QUARTERS:
    for dimension in [4,10]:
        p=RS_ROOT/'embeddings'/qlabel(y,q)/'rs_l'/f'dimension={dimension}'/'regularization_validation.parquet'
        d=pd.read_parquet(p).sort_values('validation_rmse').reset_index(drop=True); d['year']=y; d['quarter']=q; d['dimension']=dimension; curve_rows.append(d)
curves=pd.concat(curve_rows,ignore_index=True); winners=curves.sort_values('validation_rmse').groupby(['year','quarter','dimension']).head(2)
margins=winners.groupby(['year','quarter','dimension']).validation_rmse.apply(lambda x:x.iloc[1]-x.iloc[0]).rename('winner_margin').reset_index()
save(curves,ROBUST_ROOT/'rs_l'/'all_cv_curves.parquet'); save(margins,ROBUST_ROOT/'rs_l'/'winner_margins.parquet')
display(curves.groupby(['dimension','regularization'],as_index=False).validation_rmse.mean()); display(margins.groupby('dimension').winner_margin.describe())


## 8. Word2Vec boundary and convergence audit

The completed baseline selected the largest tested context radius and often reached the epoch cap. This cell summarizes those facts. A versioned extended calibration is intentionally kept separate from the baseline and should be run only after reviewing this audit.


In [ ]:
calibration=pd.read_parquet(W2V_ROOT/'calibration'/'radius_summary.parquet'); inventory=pd.read_parquet(W2V_ROOT/'embeddings'/'fit_inventory.parquet')
w2v_audit=inventory.groupby('dimension',as_index=False).agg(fits=('quarter','size'),mean_validation_loss=('best_validation_loss','mean'),
    mean_seconds=('fit_seconds','mean'),mean_best_epoch=('best_epoch','mean'),fits_best_at_epoch_cap=('best_epoch',lambda x:int((x==8).sum())))
save(w2v_audit,ROBUST_ROOT/'word2vec'/'baseline_training_audit.parquet'); display(calibration); display(w2v_audit)
print('Boundary warning:',int(calibration.iloc[0].context_radius)==int(calibration.context_radius.max()))


## 9. Economic nearest-neighbor sanity check

For the largest 2024Q4 firms, this reports cosine-nearest firms under each 10D representation. The output supports manual business interpretation and flags collapsed or implausible neighborhoods.


In [ ]:
y,q=2024,4; label=qlabel(y,q); market=pd.read_parquet(FULL_ROOT/'processed'/label/'jkp_firm_panel.parquet').sort_values('jkp_firm_market_equity',ascending=False)
anchors=market.head(10).permco.tolist(); neighbor_rows=[]
for model in ['rs_binary','rs_ranks','rs_l_0','rs_l_min','rs_l','word2vec']:
    e=pd.read_parquet(embedding_path(model,y,q,10)); cols=sorted(c for c in e if c.startswith('embedding_'))[:10]; similarity=cosine_similarity(e[cols])
    position=pd.Series(np.arange(len(e)),index=e.permco)
    for anchor in anchors:
        if anchor not in position: continue
        i=int(position[anchor]); order=np.argsort(-similarity[i]); order=order[order!=i][:10]
        for rank,j in enumerate(order,1): neighbor_rows.append({'model':model,'anchor_permco':anchor,'anchor_name':e.iloc[i].firm_name,'rank':rank,'neighbor_permco':e.iloc[j].permco,'neighbor_name':e.iloc[j].firm_name,'cosine_similarity':similarity[i,j]})
neighbors=pd.DataFrame(neighbor_rows); save(neighbors,ROBUST_ROOT/'business_checks'/'largest_firm_neighbors_2024q4.parquet'); display(neighbors.head(60))


## 10. Robustness decision table

After the cells above run, summarize which conclusions survive, which estimates change materially, and which expensive checks should be expanded from three quarters to all 80. Do not overwrite the production benchmark until the robustness specification is explicitly accepted.
